In [1]:
import os
from pathlib import Path
import logging
import shutil
from datetime import datetime
import pandas as pd
from pyspark.sql import functions as F

import teehr
from teehr.evaluation.evaluation import RemoteReadOnlyEvaluation
from teehr.evaluation.spark_session_utils import create_spark_session
from teehr.fetching.nwm.nwm_points import nwm_to_parquet
from teehr.fetching.utils import create_periods_based_on_chunksize, get_period_start_end_times
from teehr.fetching.utils import (
    format_nwm_variable_name,
    format_nwm_configuration_metadata
)
from teehr.utils.utils import remove_dir_if_exists
from teehr.fetching.const import NWM_VARIABLE_MAPPER

import dask
dask.config.set({"distributed.dashboard.link": "{JUPYTERHUB_SERVICE_PREFIX}proxy/{port}/status"})

logger = logging.getLogger()

In [2]:
LOCATION_ID_PREFIX = "nwm30"
OCONUS_STATE_NAMES = ['Alaska']
nwm_version = "nwm30"

nwm_configuration = "short_range_alaska"
output_type = "channel_rt"
variable_name = "streamflow"

- NWM v3.0: 2023-09-19 - present

#### One time: Get the NWM v3.0 location IDs to fetch and save to csv

In [3]:
%%time
spark = create_spark_session()  # For read-only
ev = teehr.RemoteReadOnlyEvaluation(spark=spark)

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:✅ Spark local configuration successful!
INFO:teehr.evaluation.spark_session_utils:Setting Hadoop's default AWS credentials provider and AWS region
INFO:teehr.evaluation.spark_session_utils:🔑 Using AWS session token from boto3
INFO:teehr.evaluation.spark_session_utils:Configuring Iceberg catalogs...
INFO:teehr.evaluation.spark_session_utils:⚙️ All settings applied. Creating Spark session...
INFO:teehr.evaluation.spark_session_utils:🎉 Spark session created successfully!
INFO:teehr.evaluation.evaluation:Using provided Spark session.
INFO:teehr.evaluation.evaluation:Active catalog set to iceberg.


CPU times: user 81.5 ms, sys: 52.1 ms, total: 134 ms
Wall time: 12.1 s


In [4]:
%%time
# Need to filter for OCONUS
included_states = ", ".join(f"'{s}'" for s in OCONUS_STATE_NAMES)
filtered_crosswalks_sdf = ev.location_crosswalks.add_attributes(
    attr_list=["state_name"]
).filter(
    filters=[
        {
            "column": "secondary_location_id",
            "operator": "like",
            "value": f"{LOCATION_ID_PREFIX}-%"
        },
        f"state_name IN ({included_states})"
    ]
).to_sdf()
stripped_ids = [
    row[0].split("-")[1]
    for row in filtered_crosswalks_sdf.select("secondary_location_id").collect()
]
len(stripped_ids)

INFO:teehr.evaluation.tables.base_table:Initializing Table for table: location_crosswalks.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.location_crosswalks.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.location_crosswalks.
INFO:teehr.evaluation.views.location_attributes_view:Computing pivoted attributes view
INFO:teehr.evaluation.tables.generic_table:Getting table: location_attributes.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: location_attributes.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.location_attributes.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.location_attributes.
INFO:teehr.evaluation.dataframe_base:Setting filter [{'column': 'secondary_location_id', 'operator': 'like', 'value': 'nwm30-%'}, "state_name IN ('Hawaii')"].


CPU times: user 22.9 ms, sys: 151 μs, total: 23 ms
Wall time: 6.13 s


58

In [5]:
df = pd.DataFrame({"nwm_id": stripped_ids})
df.to_csv("/data/data_processing/backfill-nwm/nwm_alaska_ids.csv", index=False)

In [7]:
ev.spark.stop()

#### Create a local Evaluation to store the cached parquet files

In [8]:
%%time
dir_path =  "/data/data_processing/backfill-nwm/nwm-alaska-teehr-warehouse"

spark = create_spark_session()
ev = teehr.LocalReadWriteEvaluation(
    spark=spark,
    dir_path=dir_path,
    create_dir=True
)

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:✅ Spark local configuration successful!
INFO:teehr.evaluation.spark_session_utils:Setting Hadoop's default AWS credentials provider and AWS region
INFO:teehr.evaluation.spark_session_utils:🔑 Using AWS session token from boto3
INFO:teehr.evaluation.spark_session_utils:Configuring Iceberg catalogs...
INFO:teehr.evaluation.spark_session_utils:⚙️ All settings applied. Creating Spark session...
INFO:teehr.evaluation.spark_session_utils:🎉 Spark session created successfully!
INFO:teehr.evaluation.evaluation:Using provided Spark session.
INFO:teehr.evaluation.evaluation:Creating directory /data/data_processing/backfill-nwm/nwm-hawaii-teehr-warehouse.
INFO:teehr.utilities.apply_migrations:✅ Created schema: local.schema_evolution
INFO:teehr.utilities.apply_migrations:✅ Created table: local.schema_evolution.schema_version_history
INFO:teehr.utilities.apply_migrations:Appl

CPU times: user 94.9 ms, sys: 52 ms, total: 147 ms
Wall time: 14.3 s


Get the CONUS NWM IDs (these were pre-prepared from the warehouse using the above query)

In [9]:
df_nwm_ids = pd.read_csv("/data/data_processing/backfill-nwm/nwm_alaska_ids.csv")
stripped_ids = df_nwm_ids["nwm_id"].tolist()

In [10]:
ev_variable_name = format_nwm_variable_name(variable_name)
ev_config = format_nwm_configuration_metadata(
    nwm_config_name=nwm_configuration,
    nwm_version=nwm_version
)
nwm_cache_dir = Path(
    ev.cache_dir,
    "fetching",
    "nwm"
)
kerchunk_cache_dir = Path(
    ev.cache_dir,
    "fetching",
    "kerchunk"
)

INFO:teehr.fetching.utils:Getting schema variable name for streamflow.
INFO:teehr.fetching.utils:Formatting configuration name for short_range_hawaii.


In [11]:
start_date = datetime(2023, 9, 19, 0)  # 2023-09-19 (start of nwm v3.0)
end_date = datetime(2026, 4, 22, 0)   # 2025-11-10 22:00 (the earliest non-null reference_time in secondary)
chunk_by = "week"  # Describes the chunk size between start/end date

In [12]:
# Start up a local Dask cluster
from dask.distributed import Client
client = Client(n_workers=12)  # only using 4 workers by default?
client

INFO:distributed.scheduler:State start
INFO:distributed.diskutils:Found stale lock file and directory '/tmp/dask-scratch-space/scheduler-5qtu3ek3', purging
INFO:distributed.diskutils:Found stale lock file and directory '/tmp/dask-scratch-space/scheduler-ur3qmkcb', purging
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:39913
INFO:distributed.scheduler:  dashboard at:  /user/samlamont/backfill-pr-short-range/proxy/8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:39029'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:42991'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:32997'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:32871'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:36885'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:43823'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:44461

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/samlamont/backfill-pr-short-range/proxy/8787/status,
Dashboard: /user/samlamont/backfill-pr-short-range/proxy/8787/status,Workers: 12
Total threads: 24,Total memory: 124.44 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:39913,Workers: 0
Dashboard: /user/samlamont/backfill-pr-short-range/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:44305,Total threads: 2
Dashboard: /user/samlamont/backfill-pr-short-range/proxy/40103/status,Memory: 10.37 GiB
Nanny: tcp://127.0.0.1:39029,


Note: `chunk_by` doesn't have much meaning here, unless we want to do something with the cached files each iteration. <br>
The actual processing is done in chunks governed by:
- `process_by_z_hour` - chunks by reference time
- `step_size` - if above is False, an integer number of timesteps

In [13]:
%%time
periods = create_periods_based_on_chunksize(
    start_date=start_date,
    end_date=end_date,
    chunk_by=chunk_by
)

for i, period in enumerate(periods):

    # if i <= 2:  # already processed this batch
    #     continue 
    
    if period is not None:
        dts = get_period_start_end_times(
            period=period,
            start_date=start_date,
            end_date=end_date
        )
    else:
        dts = {"start_dt": start_date, "end_dt": end_date}

    logger.info("Fetching NWM data and writing to cache")
    nwm_to_parquet(
        configuration=nwm_configuration,
        output_type=output_type,
        variable_name=variable_name,
        start_date=dts["start_dt"],
        end_date=dts["end_dt"],
        location_ids=stripped_ids,
        json_dir=kerchunk_cache_dir,
        output_parquet_dir=Path(
            nwm_cache_dir,
            ev_config["name"],
            ev_variable_name
        ),
        nwm_version=nwm_version,
        variable_mapper=NWM_VARIABLE_MAPPER,
        kerchunk_method="auto",
        process_by_z_hour=False,
        stepsize=1296,
        overwrite_output=True
    )    

INFO:root:Fetching NWM data and writing to cache
INFO:teehr.fetching.nwm.nwm_points:Fetching short_range_hawaii data. Version: nwm30
INFO:teehr.fetching.utils:Limiting the start date to z-hour: 0.
INFO:teehr.fetching.utils:Limiting the end date to z-hour: 23.
INFO:root:Fetching NWM data and writing to cache
INFO:teehr.fetching.nwm.nwm_points:Fetching short_range_hawaii data. Version: nwm30
INFO:teehr.fetching.utils:Limiting the start date to z-hour: 0.
INFO:teehr.fetching.utils:Limiting the end date to z-hour: 23.
INFO:root:Fetching NWM data and writing to cache
INFO:teehr.fetching.nwm.nwm_points:Fetching short_range_hawaii data. Version: nwm30
INFO:teehr.fetching.utils:Limiting the start date to z-hour: 0.
INFO:teehr.fetching.utils:Limiting the end date to z-hour: 23.
INFO:root:Fetching NWM data and writing to cache
INFO:teehr.fetching.nwm.nwm_points:Fetching short_range_hawaii data. Version: nwm30
INFO:teehr.fetching.utils:Limiting the start date to z-hour: 0.
INFO:teehr.fetching.uti

CPU times: user 45min 50s, sys: 2min 47s, total: 48min 37s
Wall time: 1h 46min 23s


##### Errors:
(Skipped 20240216) ValueError: There was a problem reading s3://ciroh-nwm-zarr-copy/national-water-model/nwm.20240216/short_range_alaska/nwm.t15z.short_range.channel_rt.f032.alaska.nc.json

In [ ]:
18 * 24

In [ ]:
432 * 3

In [ ]:
parquet_cache_dir = Path(
    ev.cache_dir,
    "fetching",
    "nwm",
    ev_config["name"],
    ev_variable_name    
)
parquet_cache_dir

In [ ]:
cached_sdf = ev.spark.read.parquet(parquet_cache_dir.as_posix())

In [ ]:
cached_sdf.show(n=6, truncate=False)

In [ ]:
cached_sdf.select(F.min("reference_time")).show(), cached_sdf.select(F.max("reference_time")).show()

In [ ]:
cached_sdf.select("location_id").distinct().count()

There are 83 locations for Puerto Rico